# Imports

In [2]:
import warnings

warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
from fredapi import Fred
import yfinance as yf

from dotenv import load_dotenv

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.options.display.float_format = "{:,.4f}".format

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

END_DATE = pd.Timestamp.today().normalize()
START_DATE = END_DATE - pd.DateOffset(years=3)

USE_ONLINE_DATA = True
load_dotenv()

print(f"Analysis window: {START_DATE.date()} to {END_DATE.date()}")
fred = Fred(os.getenv("FRED_API_KEY"))

Analysis window: 2023-06-01 to 2026-06-01


In [3]:
FRED_SERIES = {
    "DGS2": "2Y Treasury yield",
    "DGS5": "5Y Treasury yield",
    "DGS10": "10Y Treasury yield",
    "DGS30": "30Y Treasury yield",
    "BAMLC0A0CM": "ICE BofA US Corporate OAS",
    "BAMLH0A0HYM2": "ICE BofA US High Yield OAS",
    "VIXCLS": "CBOE VIX close",
    "T10YIE": "10Y breakeven inflation",
}

In [4]:
ETF_TICKERS = ["AGG", "LQD", "HYG", "TLT"]
PORTFOLIO_WEIGHTS = {"AGG": 0.40, "LQD": 0.25, "HYG": 0.20, "TLT": 0.15}

print(f"Total weights: {sum(PORTFOLIO_WEIGHTS.values())}")

Total weights: 1.0


# Utils

In [5]:
def fetch_fred_series(series_id, start=START_DATE, end=END_DATE):
    data_series = fred.get_series(series_id)

    df = pd.DataFrame(data_series)
    df.columns = [series_id]
    df = df.loc[(df.index >= pd.to_datetime(start)) & (df.index <= pd.to_datetime(end))]
    return df


def fetch_fed_panel(series_dict, start, end):
    frames = []

    for series_id in series_dict:
        frames.append(fetch_fred_series(series_id, start, end))

    panel = pd.concat(frames, axis=1).sort_index()

    panel.index = pd.to_datetime(panel.index).tz_localize(None)

    panel = panel.asfreq("B").ffill()
    panel = panel.dropna(how="all")

    return panel


def _ar1(n, sigma, phi, rng):
    eps = rng.normal(0, sigma, n)
    out = np.zeros(n)

    for i in range(1, n):
        out[i] = phi * out[i - 1] + eps[i]

    return out


def engineer_factor_changes(levels):

    levels = levels.copy().sort_index()
    out = pd.DataFrame(index=levels.index)

    for col in ["DGS2", "DGS5", "DGS10", "DGS30"]:
        if col in levels.columns:
            out[f"d_{col}_bp"] = levels[col].diff() * 100

    required_yields = ["DGS2", "DGS5", "DGS10", "DGS30"]

    if all(c in levels.columns for c in required_yields):
        levels["Rate_Level"] = levels[required_yields].mean(axis=1)
        levels["Slope_10Y_2Y"] = levels["DGS10"] - levels["DGS2"]
        levels["Curvature_5Y"] = 2 * levels["DGS5"] - levels["DGS2"] - levels["DGS10"]
        out["d_Rate_Level_bp"] = levels["Rate_Level"].diff() * 100
        out["d_10y2y_Slope_bp"] = levels["Slope_10Y_2Y"].diff() * 100
        out["d_Curvature_bp"] = levels["Curvature_5Y"].diff() * 100

    for col in ["BAMLC0A0CM", "BAMLH0A0HYM2"]:
        if col in levels.columns:
            out[f"d_{col}_bp"] = levels[col].diff() * 100

    if "VIXCLS" in levels.columns:
        out["d_VIX_points"] = levels["VIXCLS"].diff()

    if "T10YIE" in levels.columns:
        out["d_T10YIE_bp"] = levels["T10YIE"].diff() * 100

    return out.dropna(how="all")


def download_etf_prices(tickers, start, end):

    raw = yf.download(
        tickers,
        start=start.strftime("%Y-%m-%d"),
        end=(end + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
        auto_adjust=True,
        progress=False,
        threads=True,
    )

    if "Close" in raw.columns.get_level_values(0):
        prices = raw["Close"].copy()
    elif "Adj Close" in raw.columns.get_level_values(0):
        prices = raw["Adj Close"].copy()

    prices.index = pd.to_datetime(prices.index).tz_localize(None)
    prices = prices.sort_index().ffill().dropna(how="all")

    return prices


def portfolio_returns_from_prices(prices, weights):
    rets = prices.pct_change().dropna(how="all")
    available = [ticker for ticker in weights if ticker in rets.columns]

    w = pd.Series({ticker: weights[ticker] for ticker in available}, dtype=float)
    w = w / w.sum()

    port = rets[available].mul(w, axis=1).sum(axis=1).rename("Portfolio_Return")

    return port, rets[available]

# Loading Factors

In [6]:
market_levels = fetch_fed_panel(FRED_SERIES, START_DATE, END_DATE)
market_levels = market_levels.dropna(
    thresh=max(2, int(0.60 * len(market_levels))), axis=1
)

print(market_levels.shape)
display(market_levels.tail())

(782, 8)


,DGS2,DGS5,DGS10,DGS30,BAMLC0A0CM,BAMLH0A0HYM2,VIXCLS,T10YIE
2026-05-25,4.1300,4.2700,4.5600,5.0700,0.7400,2.7400,16.5900,2.4000
2026-05-26,4.0100,4.1900,4.5000,5.0300,0.7400,2.7200,17.0100,2.4000
2026-05-27,4.0000,4.1700,4.4800,5.0100,0.7400,2.7100,16.2900,2.3900
2026-05-28,3.9900,4.1500,4.4500,4.9800,0.7300,2.7200,15.7400,2.3900
2026-05-29,3.9900,4.1500,4.4500,4.9800,0.7300,2.7200,15.7400,2.3800


In [10]:
factor_changes = engineer_factor_changes(market_levels)
PCA_FACTOR_COLS = [
    "d_DGS2_bp",
    "d_DGS5_bp",
    "d_DGS10_bp",
    "d_DGS30_bp",
    "d_BAMLC0A0CM_bp",
    "d_BAMLH0A0HYM2_bp",
    "d_VIX_points",
    "d_T10YIE_bp",
]

REGRESSION_FACTOR_COLS = [
    "d_Rate_Level_bp",
    "d_10y2y_Slope_bp",
    "d_Curvature_bp",
    "d_BAMLC0A0CM_bp",
    "d_BAMLH0A0HYM2_bp",
    "d_VIX_points",
    "d_T10YIE_bp",
]

PCA_FACTOR_COLS = [
    c
    for c in PCA_FACTOR_COLS
    if c in factor_changes.columns and factor_changes[c].std() > 1e-12
]
REGRESSION_FACTOR_COLS = [
    c
    for c in REGRESSION_FACTOR_COLS
    if c in factor_changes.columns and factor_changes[c].std() > 1e-12
]

display(
    factor_changes[
        PCA_FACTOR_COLS
        + [c for c in REGRESSION_FACTOR_COLS if c not in PCA_FACTOR_COLS]
    ]
    .describe()
    .T
)

,count,mean,std,min,25%,50%,75%,max
d_DGS2_bp,781.0000,-0.0435,5.7758,-28.0000,-3.0000,0.0000,3.0000,23.0000
d_DGS5_bp,781.0000,0.0576,5.9894,-24.0000,-4.0000,0.0000,4.0000,24.0000
d_DGS10_bp,781.0000,0.1076,5.5198,-19.0000,-3.0000,0.0000,4.0000,19.0000
d_DGS30_bp,781.0000,0.1460,5.1413,-16.0000,-3.0000,0.0000,3.0000,17.0000
d_BAMLC0A0CM_bp,781.0000,-0.0883,1.3582,-5.0000,-1.0000,0.0000,1.0000,10.0000
d_BAMLH0A0HYM2_bp,781.0000,-0.2471,7.2311,-38.0000,-4.0000,0.0000,3.0000,59.0000
d_VIX_points,781.0000,0.0001,1.7968,-18.7100,-0.6200,-0.0700,0.5300,15.2900
d_T10YIE_bp,781.0000,0.0282,2.2813,-16.0000,-1.0000,0.0000,1.0000,13.0000
d_Rate_Level_bp,781.0000,0.0669,5.2540,-21.2500,-3.2500,0.0000,3.2500,20.0000
d_10y2y_Slope_bp,781.0000,0.1511,3.6191,-13.0000,-2.0000,0.0000,2.0000,14.0000


# Loading Portfolio Returns

In [11]:
etf_prices = download_etf_prices(ETF_TICKERS, START_DATE, END_DATE)
portfolio_returns, constituent_returns = portfolio_returns_from_prices(
    etf_prices, PORTFOLIO_WEIGHTS
)

print(portfolio_returns.dropna().shape[0])
display(portfolio_returns.tail().to_frame())

750


,Portfolio_Return
Date,
2026-05-22,0.0017
2026-05-26,0.0035
2026-05-27,0.0009
2026-05-28,0.0026
2026-05-29,0.0007
